# 第四阶段：STL（标准模板库）

## 实验 1：`std::string` —— 拥有字符序列

前三个阶段已经建立了对象、生命周期与 RAII 的概念。本实验把这些概念应用到第一个标准库类型 `std::string`：它不只是更方便的 `char*`，而是一个拥有字符存储、自动管理生命周期的值类型。

完成后你应该能够：

- 从所有权角度解释 `std::string`；
- 区分 `size()` 与 `capacity()`，理解扩容但不依赖具体增长策略；
- 解释修改字符串为何可能使 `data()`、`c_str()`、引用和迭代器失效；
- 正确地把 `c_str()` 作为临时借用传给 C API；
- 解释为什么任意字节数据的 C ABI 应同时传递 pointer 与 length。

In [1]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <cassert>
#include <cstdint>
#include <cstring>
#include <iomanip>
#include <iostream>
#include <string>

### 1. 先观察一个 `std::string` 对象

`size()` 是当前字符串包含的元素数；`capacity()` 是在下一次扩容前至少可以容纳的元素数；`data()` 指向连续存储的字符序列。容量和地址由标准库实现与运行过程决定，因此每台机器上的具体输出都可能不同。

In [2]:
// 本步骤：通过代码演示“先观察一个 std::string 对象”并观察结果。
// 只读借用 string，集中输出值、逻辑长度、容量和存储地址。
void print_string(const std::string& value)
{
    std::cout
        << "value    = " << value << '\n'
        << "size     = " << value.size() << '\n'
        << "capacity = " << value.capacity() << '\n'
        << "data     = "
        << static_cast<const void*>(value.data()) << '\n';
}

// 创建 owner 并调用观察函数，不复制底层字符。
{
    std::string name = "Kotlin";
    print_string(name);

    // 验证 size、下标和边界检查访问得到一致结果。
    assert(name.size() == 6);
    assert(name[0] == 'K');
    assert(name.at(5) == 'n');
}

value    = Kotlin
size     = 6
capacity = 22
data     = 0x16fb184e8


`operator[]` 不检查下标，适合调用方已经保证范围正确的情况；`at()` 会检查范围并在越界时抛出 `std::out_of_range`。两者返回的都是字符串内部元素，不会创建新的字符串。

从所有权看，`name` 负责其字符存储的获取、调整与释放：

```text
std::string name ── owns ──> contiguous character storage
       │
       └─ scope ends → ~string() → release storage
```

因此 `std::string` 本身就是 RAII 类型。短字符串有时会直接存储在对象内部，这种优化称为 Small String Optimization（SSO）；标准并不要求它存在，也不规定阈值，所以程序不能依赖它。

### 2. 复制得到独立的值

复制 `std::string` 会得到内容相同、生命周期独立的字符串。修改副本不会修改原对象，这正是值语义。

In [3]:
// 本步骤：通过代码演示“复制得到独立的值”并观察结果。
{
    std::string original = "Kotlin";
    std::string copy = original;

    copy += "/Native";

    std::cout << "original = " << original << '\n';
    std::cout << "copy     = " << copy << '\n';

    assert(original == "Kotlin");
    assert(copy == "Kotlin/Native");
}

original = Kotlin
copy     = Kotlin/Native


函数只需读取字符串且不需要保存副本时，可以使用 `const std::string&`，如前面的 `print_string()`。它表达“只读借用”，避免一次不必要的复制；调用方必须保证被借用的字符串在函数调用期间仍然存活。

### 3. `size()`、`capacity()` 与 `reserve()`

追加字符会增加 `size()`。当现有存储不够时，字符串需要申请更大的存储并释放旧存储，这时 `capacity()` 会增长。增长倍数是实现细节，不能假设容量一定翻倍。

如果已知大致的最终大小，可用 `reserve()` 提前请求容量。`reserve(n)` 只改变容量，不会添加字符，也不会改变 `size()`。

In [4]:
// 本步骤：通过代码演示“size()、capacity() 与 reserve()”并观察结果。
{
    std::string sdk_name = "Kotlin";
    const auto original_size = sdk_name.size();

    sdk_name.reserve(original_size + 32);

    assert(sdk_name.size() == original_size);
    assert(sdk_name.capacity() >= original_size + 32);

    sdk_name += "/Native";
    print_string(sdk_name);

    assert(sdk_name == "Kotlin/Native");
}

value    = Kotlin/Native
size     = 13
capacity = 47
data     = 0x12877edd0


`capacity()` 与后续 `vector` 实验中的概念相同：容器可以把逻辑元素数量与已申请的存储空间分开，从而减少频繁分配。`reserve()` 是性能提示，不改变字符串的值。

### 4. 重分配会使内部借用失效

`data()` 返回指向字符串内部连续存储的指针。下面先保存该指针，再请求一个大于现有容量的新容量。容量确实增长时会发生重分配，旧指针、引用和迭代器随即失效。

为了只观察地址数值，示例在重分配前先把地址转换成整数保存；重分配后不再读取旧指针。

In [5]:
// 本步骤：通过代码演示“重分配会使内部借用失效”并观察结果。
{
    std::string value = "Kotlin/Native";
    const char* borrowed = value.data();
    const auto old_address =
        reinterpret_cast<std::uintptr_t>(borrowed);
    const auto old_capacity = value.capacity();

    value.reserve(old_capacity + 100);

    std::cout << std::boolalpha
              << "capacity grew = "
              << (value.capacity() > old_capacity) << '\n'
              << "address changed = "
              << (old_address != reinterpret_cast<std::uintptr_t>(value.data()))
              << '\n';

    assert(value.capacity() > old_capacity);

    // borrowed 已失效：此后不能读取 *borrowed。
}

capacity grew = true
address changed = true


不要把示例误解为“只有地址数值变化才算失效”。判断一个借用能否继续使用，应依据 API 的失效规则，而不是事后比较地址；分配器甚至可能复用同一个数值地址。安全规则是：取得 `data()`、`c_str()`、元素引用或迭代器后，不要跨越可能修改字符串的操作长期保存它们，需要时重新获取。

### 5. `c_str()` 是通往 C API 的临时桥梁

`c_str()` 返回以空字符结尾的 `const char*`，可以传给只在调用期间读取文本的 C API。该指针不拥有内存，不能释放，也不能比原字符串活得更久。

In [6]:
// 本步骤：通过代码演示“cstr() 是通往 C API 的临时桥梁”并观察结果。
void inspect_c_text(const char* text)
{
    std::cout << "C text   = " << text << '\n';
    std::cout << "C length = " << std::strlen(text) << '\n';
}

{
    std::string name = "Kotlin/Native";

    // 借用只跨越这一次调用；name 在调用期间保持存活且不被修改。
    inspect_c_text(name.c_str());
}

C text   = Kotlin/Native
C length = 13


典型的错误是从局部字符串返回 `c_str()`：

```cpp
const char* bad_name()
{
    std::string local = "Kotlin";
    return local.c_str(); // local 析构后，返回值立即悬空
}
```

如果调用方需要长期保存内容，应复制字符串，或由 API 明确定义内存分配与释放协议。

### 6. `std::string` 的长度不等于 C 字符串长度

`std::string` 可以包含嵌入的 `\0`，其 `size()` 仍记录完整元素数量；传统 C 字符串函数把第一个 `\0` 当作结尾。下面使用 `(pointer, count)` 构造函数保留三个字节。

In [7]:
// 本步骤：通过代码演示“std::string 的长度不等于 C 字符串长度”并观察结果。
{
    // 准备包含嵌入 NUL 的三个原始字节，并按明确长度构造 owner。
    const char raw[] = {'A', '\0', 'B'};
    std::string payload(raw, 3);

    // 对比 string 记录的完整长度和 strlen 看到的 C 字符串长度。
    std::cout << "string size = " << payload.size() << '\n';
    std::cout << "strlen      = " << std::strlen(payload.c_str()) << '\n';
    std::cout << "bytes       =";

    // 逐字节输出，证明 NUL 后面的 B 仍属于 string。
    for (unsigned char byte : payload)
    {
        std::cout << " 0x"
                  << std::hex << std::setw(2) << std::setfill('0')
                  << static_cast<int>(byte);
    }

    std::cout << std::dec << '\n';

    // 用断言固定两种长度语义的差异。
    assert(payload.size() == 3);
    assert(std::strlen(payload.c_str()) == 1);
}

string size = 3
strlen      = 1
bytes       = 0x41 0x00 0x42


这也是 Native SDK 设计 C ABI 时的重要区别：

```c
// 只适用于约定不含嵌入 NUL、以 NUL 结尾的文本
void sdk_set_name(const char* text);

// 可以表达任意字节序列，也能明确长度
void sdk_send_bytes(const unsigned char* data, size_t size);
```

此外，`std::string::size()` 统计的是 `char` 元素数量（通常对应编码后的字节数），不是 Unicode 字符或用户看到的字形数量。字符串编码必须由 API 契约另行约定，例如明确要求 UTF-8。

### 本实验结论

`std::string` 是拥有连续字符存储的 RAII 值类型：复制产生独立的值，析构自动释放资源。`size()` 描述当前元素数，`capacity()` 描述已预留的存储能力，具体容量增长和 SSO 都是实现细节。

`data()` 与 `c_str()` 暴露的是内部存储的借用视图，而不是所有权转移。只要原字符串析构或发生可能导致失效的修改，旧借用就不能继续使用。跨 C ABI 传递文本时还必须明确生命周期、编码和 NUL 规则；传递任意数据时应使用 pointer + length。

下一实验将引入 `std::string_view`，专门表达“不拥有字符串，只借用一段连续字符”的语义。